<div align="center">

<a href="https://colab.research.google.com/github/utkukose/human-computer-interaction-NB-lecture/blob/main/exams/midterm/MID1_mobile_task.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Midterm option 1: A task in a mobile app

**Human-Computer Interaction (BIL-429 İnsan-Bilgisayar Etkileşimi)**<br>Midterm project: A usability study of an interface. Starter Colab notebook

[Project page](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/exams/midterm) | [Course repository](https://github.com/utkukose/human-computer-interaction-NB-lecture)

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. First version: November 2026.</sub>

</div>

## How to use this starter Colab notebook

Save a copy in Drive first, with File, Save a copy in Drive. Every code cell runs as it is with an example case: Renewing a borrowed book in the app of a university library. Replace the values marked `# TODO` with the measurements of your own interface. Then run all cells again from the top, with Runtime, Run all, and use the printed values and figures in the report. Each section names the week and the lecture section that explain its method. The last section collects the numbers for the results table of the report. Keep your own data in the cells, so that the Colab notebook documents how every number of the report was obtained.

## 0. Setup

The cell loads the libraries and creates the dictionary `RESULTS`, which collects the numbers of the report.

In [ ]:
import math
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

RANDOM_STATE = 42
RESULTS = {}            # the numbers for the results table of the report
print("Setup complete.")

## 1. The task as a chain of steps

*Lecture: [Week 1, A task as a chain of steps](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-01/lecture.html#a-task-as-a-chain-of-steps)*

Each step has a time $t$ in seconds, a probability $e$ of a slip, a repair time $r$ in seconds and a probability $a$ of giving up after a slip. The probability of a flow without a slip is $P_0 = \prod_i (1 - e_i)$. The expected time is $E[T] = \sum_i (t_i + e_i r_i)$, and the probability of completing the task is $P_c = \prod_i (1 - e_i a_i)$. Estimate the values from a few observed trials of your task, and write in the report how you obtained them.

In [ ]:
# TODO: the steps of your task, with values from a few observed trials
FLOW = [
    {"step": "Open the app and sign in",    "t": 14, "e": 0.10, "r": 20, "a": 0.10},
    {"step": "Find the loans page",         "t": 9,  "e": 0.30, "r": 12, "a": 0.15},
    {"step": "Select the book",             "t": 5,  "e": 0.05, "r": 6,  "a": 0.05},
    {"step": "Renew and read the new date", "t": 6,  "e": 0.20, "r": 15, "a": 0.25},
]

def p_smooth(flow):
    """P_0: probability that no step of the flow has a slip."""
    p = 1.0
    for s in flow:
        p *= 1 - s["e"]
    return p

def expected_time(flow):
    """E[T]: expected duration in seconds when every slip is repaired."""
    return sum(s["t"] + s["e"] * s["r"] for s in flow)

def p_complete(flow):
    """P_c: probability of completing the flow; people give up only after a slip."""
    p = 1.0
    for s in flow:
        p *= 1 - s["e"] * s["a"]
    return p

steps = pd.DataFrame(FLOW)
steps["expected loss (s)"] = steps["e"] * steps["r"]
print(steps.to_string(index=False))
print(f"P_0 = {p_smooth(FLOW):.3f}, E[T] = {expected_time(FLOW):.1f} s, "
      f"P_c = {p_complete(FLOW):.3f}")
RESULTS["current flow: P_0"] = round(p_smooth(FLOW), 3)
RESULTS["current flow: E[T] (s)"] = round(expected_time(FLOW), 1)
RESULTS["current flow: P_c"] = round(p_complete(FLOW), 3)

The column *expected loss* is $e\,r$, the seconds that the slips of a step cost on average. The step with the largest loss is the first candidate for the redesign.

## 2. A heuristic evaluation

*Lecture: [Week 5, Heuristic evaluation](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#heuristic-evaluation) and [Week 5, How many evaluators](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-05/lecture.html#how-many-evaluators)*

At least three evaluators inspect the screens alone, each against the ten heuristics, and the lists are then merged. The matrix `FOUND` holds a 1 where an evaluator found a problem. `SEVERITY` holds the severity that each evaluator gives each merged problem, from 0 for none to 4 for a catastrophe. $\lambda$ is the mean of `FOUND`, and the adjustment of Lewis corrects its overestimate for a few evaluators. $n$ evaluators find the share $1 - (1 - \lambda)^n$ of all problems.

In [ ]:
# TODO: the merged list of problems, each with the heuristic that it violates
PROBLEMS = [
    ("H1", "No feedback while the renewal is sent"),
    ("H2", "The menu says Circulation instead of My loans"),
    ("H3", "No way back from the barcode scanner"),
    ("H4", "Two words for one date: due date and return date"),
    ("H5", "Renew all sits right next to Return all"),
    ("H6", "The library PIN must be typed on every visit"),
    ("H7", "No shortcut from the reminder to the loan"),
    ("H8", "Every loan repeats the full address of the library"),
    ("H9", "Error 403 appears without an explanation"),
    ("H10", "No help when a renewal is blocked"),
]
# TODO: 1 if the evaluator found the problem; one row per evaluator
FOUND = np.array([
    [1, 1, 0, 1, 1, 0, 0, 1, 1, 0],
    [1, 0, 1, 0, 1, 1, 0, 0, 1, 0],
    [0, 1, 0, 0, 1, 0, 1, 0, 1, 1],
])
# TODO: the severity that each evaluator gives each problem, from 0 to 4
SEVERITY = np.array([
    [2, 2, 3, 1, 4, 2, 1, 1, 3, 2],
    [3, 1, 3, 1, 4, 3, 1, 0, 3, 2],
    [2, 2, 2, 2, 3, 2, 2, 1, 4, 3],
])

def found_share(lam, n):
    """Share of all problems found by n evaluators."""
    return 1 - (1 - lam) ** n

def evaluators_for(goal, lam):
    """Smallest whole number of evaluators that reaches the goal share."""
    return int(math.ceil(math.log(1 - goal) / math.log(1 - lam) - 1e-9))

def lewis_adjust(M):
    """Estimate of lambda from a 0/1 matrix and its adjustment for few rows."""
    M = np.asarray(M)
    M = M[:, M.sum(axis=0) > 0]               # only problems found at all
    n = M.shape[0]
    lam = M.mean()
    gt = np.mean(M.sum(axis=0) == 1)          # share of problems found once
    adj = 0.5 * (lam - 1 / n) * (1 - 1 / n) + 0.5 * lam / (1 + gt)
    return lam, gt, adj

lam, gt, adj = lewis_adjust(FOUND)
n_eval, n_found = FOUND.shape[0], int((FOUND.sum(axis=0) > 0).sum())
share = found_share(adj, n_eval)
print(f"lambda {lam:.3f}, found once {gt:.3f}, adjusted lambda {adj:.3f}")
print(f"{n_eval} evaluators found {n_found} problems, an estimated {share:.3f} "
      f"of about {n_found / share:.0f}")
print("evaluators for 90 percent:", evaluators_for(0.90, adj))
ranked = pd.DataFrame(PROBLEMS, columns=["heuristic", "problem"])
ranked["found by"] = FOUND.sum(axis=0)
ranked["mean severity"] = SEVERITY.mean(axis=0).round(2)
ranked = ranked.sort_values(["mean severity", "found by"], ascending=False)
print(ranked.to_string(index=False))
RESULTS["problems found"] = n_found
RESULTS["adjusted lambda"] = round(adj, 3)
RESULTS["most severe problem"] = ranked.iloc[0]["problem"]

In [ ]:
# Figure: the share of the problems found against the number of evaluators
n = np.arange(1, 11)
plt.figure(figsize=(6, 3.4))
plt.plot(n, found_share(adj, n), "o-", color="#5A4BB5", label="adjusted lambda")
plt.axhline(0.9, color="#C98510", ls="--", label="90 percent")
plt.xlabel("evaluators")
plt.ylabel("share of the problems found")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Personas from a short survey

*Lecture: [Week 6, Personas](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#personas)*

A short survey asks the users of the app about their age, their use of the app and their skill with apps. It also asks for the minutes that they can spare for the task. The answers are standardised, and k-means groups similar people. The within-group sum of squares $W$ falls with every added group. The number of groups at the elbow of $W$ gives the number of personas. The example simulates 30 answers; replace them with the answers of at least 20 real users.

In [ ]:
# TODO: replace the simulated answers with the answers of your survey
rng = np.random.default_rng(RANDOM_STATE)
GROUPS = [(12, 20, 25, 4.5, 2), (10, 35, 6, 3.0, 8), (8, 58, 2, 2.0, 15)]
rows = []
for size, age, uses, skill, minutes in GROUPS:
    for _ in range(size):
        rows.append([rng.normal(age, 3), max(0.0, rng.normal(uses, 2)),
                     float(np.clip(rng.normal(skill, 0.5), 1, 5)),
                     max(1.0, rng.normal(minutes, 2))])
COLS = ["age", "uses per month", "app skill", "minutes to spare"]
SURVEY = pd.DataFrame(rows, columns=COLS).round(1)

def scale(df):
    """z-scores of each column, with the standard deviation of the column."""
    return (df - df.mean()) / df.std(ddof=0)

Z = scale(SURVEY[COLS])
W = {}
for k in range(1, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Z)
    W[k] = round(float(km.inertia_), 1)
print("within-group sum of squares W for k = 1 to 6:", W)
K = 3      # TODO: the number of personas, chosen at the elbow of W
km = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(Z)
SURVEY["group"] = km.labels_
print(SURVEY.groupby("group")[COLS].mean().round(1))
print("people per group:", SURVEY["group"].value_counts().sort_index().to_dict())
RESULTS["personas"] = K

In [ ]:
# Figure: the elbow of W and the groups of the survey
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(list(W), list(W.values()), "o-", color="#5A4BB5")
ax[0].set_xlabel("number of groups k")
ax[0].set_ylabel("W")
for g in range(K):
    part = SURVEY[SURVEY["group"] == g]
    ax[1].scatter(part["age"], part["uses per month"], label=f"group {g}", s=18)
ax[1].set_xlabel("age")
ax[1].set_ylabel("uses per month")
ax[1].legend()
plt.tight_layout()
plt.show()

Give each group a name, a goal and a context in the report. The table of means is the evidence behind each persona.

## 4. Predicted task times with the keystroke-level model

*Lecture: [Week 6, Keystroke-level model](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-06/lecture.html#keystroke-level-model)*

The keystroke-level model (KLM) predicts the time of an expert who makes no errors. A method is a sequence of operators. Here, M stands for a mental preparation and T for a tap on the screen. K is a key of the on-screen keyboard, and W is the wait for the system. The operator times differ between the personas. Write the sequences of the current design and of the redesign, and estimate the times of the personas from a few observations or from the times of Week 6.

In [ ]:
# Operator times in s: M mental preparation, T a tap, K a key of the
# on-screen keyboard; W, the response time of the app, is the same for all
PERSONAS = {   # TODO: one entry per persona, from the groups above
    "Deniz, the daily student": {"M": 1.2, "T": 0.6, "K": 0.35},
    "Selin, the busy lecturer": {"M": 1.35, "T": 0.8, "K": 0.45},
    "Ahmet, the retired reader": {"M": 2.0, "T": 1.1, "K": 0.7},
}
WAIT = 1.5     # TODO: the measured response time of the app (s)
CURRENT_OPS = "MT" + "W" + "MTMT" + "MTKKKK" + "W" + "MT" + "MT"   # TODO
REDESIGN_OPS = "MT" + "W" + "MT" + "MT"                            # TODO

def klm_time(ops, persona):
    """Predicted time of an operator sequence for one persona (s)."""
    times = dict(PERSONAS[persona], W=WAIT)
    return round(sum(times[c] for c in ops), 2)

klm = pd.DataFrame([(p, klm_time(CURRENT_OPS, p), klm_time(REDESIGN_OPS, p))
                    for p in PERSONAS],
                   columns=["persona", "current (s)", "redesign (s)"])
klm["saving (s)"] = (klm["current (s)"] - klm["redesign (s)"]).round(2)
print(klm.to_string(index=False))
for _, row in klm.iterrows():
    RESULTS[f"KLM saving, {row['persona']} (s)"] = row["saving (s)"]

## 5. The redesign as a state machine

*Lecture: [Week 7, State machines](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-07/lecture.html#state-machines)*

A prototype is a set of screens and of the events that move between them. `RULES_CURRENT` and `RULES_REDESIGN` map a screen and an event to the next screen. A breadth-first search gives the fewest events from one screen to another, and `no_way_home` lists the screens from which the home screen cannot be reached.

In [ ]:
# TODO: the screens of your current app and of the redesign, with their events
RULES_CURRENT = {("Home", "menu"): "Menu", ("Menu", "account"): "Account",
                 ("Menu", "home"): "Home", ("Account", "loans"): "Loans",
                 ("Account", "back"): "Menu", ("Loans", "book"): "Book",
                 ("Loans", "back"): "Account", ("Book", "renew"): "Renewed",
                 ("Book", "back"): "Loans", ("Renewed", "done"): "Home",
                 ("Home", "scan"): "Scanner"}
RULES_REDESIGN = {("Home", "loans"): "Loans", ("Home", "scan"): "Scanner",
                  ("Scanner", "close"): "Home", ("Loans", "renew"): "Renewed",
                  ("Loans", "home"): "Home", ("Renewed", "home"): "Home"}

def neighbours(rules, state):
    """The screens that one event leads to from state."""
    return [nxt for (s, e), nxt in rules.items() if s == state]

def shortest(rules, start, goal):
    """Fewest events from start to goal, by breadth-first search."""
    dist, queue = {start: 0}, deque([start])
    while queue:
        s = queue.popleft()
        for nxt in neighbours(rules, s):
            if nxt not in dist:
                dist[nxt] = dist[s] + 1
                queue.append(nxt)
    return dist.get(goal)

def no_way_home(rules, home):
    """States from which no sequence of events leads back to home."""
    states = {s for s, _ in rules} | set(rules.values())
    return sorted(s for s in states
                  if s != home and shortest(rules, s, home) is None)

for name, rules in [("current", RULES_CURRENT), ("redesign", RULES_REDESIGN)]:
    print(f"{name}: {shortest(rules, 'Home', 'Renewed')} events to the renewal, "
          f"no way home from {no_way_home(rules, 'Home')}")
RESULTS["events to the goal, current"] = shortest(RULES_CURRENT, "Home", "Renewed")
RESULTS["events to the goal, redesign"] = shortest(RULES_REDESIGN, "Home", "Renewed")
RESULTS["screens without a way home, current"] = len(
    no_way_home(RULES_CURRENT, "Home"))

Real users do not always take the shortest way. A Markov model gives each move the share of the users who take it, and the expected number of taps to the goal follows from a linear system. In the example, the users who open the scanner of the current app can only leave it by closing the app, which costs a new start from the home screen.

In [ ]:
# TODO: the share of the users who take each move, from observations
P_CURRENT = {"Home": {"Menu": 0.85, "Scanner": 0.15},
             "Scanner": {"Home": 1.0},      # leaving means a restart of the app
             "Menu": {"Account": 0.7, "Home": 0.3},
             "Account": {"Loans": 0.8, "Menu": 0.2},
             "Loans": {"Book": 0.9, "Account": 0.1},
             "Book": {"Renewed": 0.9, "Loans": 0.1}}
P_REDESIGN = {"Home": {"Loans": 0.95, "Scanner": 0.05},
              "Scanner": {"Home": 1.0},
              "Loans": {"Renewed": 0.95, "Home": 0.05}}

def expected_taps(P, goal):
    """Expected taps to reach goal from each state of a Markov model P."""
    states = [s for s in P if s != goal]
    idx = {s: i for i, s in enumerate(states)}
    Q = np.zeros((len(states), len(states)))
    for s in states:
        for nxt, p in P[s].items():
            if nxt != goal:
                Q[idx[s], idx[nxt]] += p
    t = np.linalg.solve(np.eye(len(states)) - Q, np.ones(len(states)))
    return {s: round(float(x), 3) for s, x in zip(states, t)}

taps_current = expected_taps(P_CURRENT, "Renewed")["Home"]
taps_redesign = expected_taps(P_REDESIGN, "Renewed")["Home"]
print(f"expected taps from Home: current {taps_current}, redesign {taps_redesign}")
RESULTS["expected taps, current"] = taps_current
RESULTS["expected taps, redesign"] = taps_redesign

Finally, the redesign is written as a chain of steps with the values that you expect, so that $P_0$, $E[T]$ and $P_c$ of both designs can be compared. Justify each value in the report, for example with a heuristic that the redesign no longer violates.

In [ ]:
FLOW_REDESIGN = [   # TODO: the steps of your redesign with expected values
    {"step": "Open the app, signed in", "t": 6, "e": 0.05, "r": 20, "a": 0.10},
    {"step": "Tap My loans",            "t": 4, "e": 0.10, "r": 8,  "a": 0.10},
    {"step": "Renew and read the date", "t": 5, "e": 0.08, "r": 10, "a": 0.15},
]
for name, flow in [("current", FLOW), ("redesign", FLOW_REDESIGN)]:
    print(f"{name}: P_0 {p_smooth(flow):.3f}, E[T] {expected_time(flow):.1f} s, "
          f"P_c {p_complete(flow):.3f}")
RESULTS["redesign: P_0"] = round(p_smooth(FLOW_REDESIGN), 3)
RESULTS["redesign: E[T] (s)"] = round(expected_time(FLOW_REDESIGN), 1)
RESULTS["redesign: P_c"] = round(p_complete(FLOW_REDESIGN), 3)

## 6. Results for the report

The cell collects the numbers that the sections above stored in `RESULTS`. Copy them into the results table of the report, and explain each one in the text. A value that changes the design deserves a figure.

In [ ]:
import pandas as pd

print(pd.Series(RESULTS, name="value", dtype=object).to_string())

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>